In [3]:
import torch
import torch.nn as nn
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import os
from typing import List

In [2]:
AUDIO_DIR = "../../data_processing/separate_scripts/golden_audio"
LABEL_DIR = "../../data_processing/separate_scripts/golden_breaks"
INDEX_SET = [0, 6, 16, 19]

In [ ]:
class BreakDataset(Dataset):
    def __init__(self, indices: List[int] = INDEX_SET, audio_dir: str = AUDIO_DIR, label_dir: str = LABEL_DIR):
        self.indices = indices
        self.audio_dir = audio_dir
        self.label_dir = label_dir

        self.data = []
        for idx in indices:
            # Load audio and convert to spectrogram
            audio_path = os.path.join(audio_dir, f"{idx}.mp3")
            waveform, sr = torchaudio.load(audio_path)

            # Ensure mono channel - explicitly convert to single channel
            waveform = waveform.mean(dim=0, keepdim=True)

            # Compute mel spectrogram with adjusted parameters
            mel_spec = torchaudio.transforms.MelSpectrogram(
                sample_rate=sr,
                n_fft=2205,
                hop_length=int(sr * 0.02),
                n_mels=128,  # Ensure this matches your expected input size
                f_min=20,
                f_max=8000
            )(waveform)

            # Load labels
            label_path = os.path.join(label_dir, f"{idx}.csv")
            df = pd.read_csv(label_path)
            labels = torch.tensor(df['break'].values, dtype=torch.bool)

            # Ensure spectrogram and labels have same length
            min_len = min(mel_spec.shape[2], len(labels))
            
            
            mel_spec = mel_spec[:, :, :min_len]
            labels = labels[:min_len]

            self.data.append((mel_spec, labels))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    # Separate spectrograms and labels
    specs, labels = zip(*batch)

    # Get lengths of each sequence
    lengths = [spec.shape[2] for spec in specs]
    max_len = max(lengths)

    # Pad spectrograms
    padded_specs = []
    padded_labels = []

    for spec, label in zip(specs, labels):
        # Pad spectrogram
        curr_len = spec.shape[2]
        padding = max_len - curr_len
        padded_spec = torch.nn.functional.pad(spec, (0, padding))  # Pad the time dimension
        padded_specs.append(padded_spec)

        # Pad labels
        padded_label = torch.nn.functional.pad(label, (0, padding))
        padded_labels.append(padded_label)

    # Stack into batches
    padded_specs = torch.stack(padded_specs)
    padded_labels = torch.stack(padded_labels)

    return padded_specs, padded_labels, torch.tensor(lengths)

In [25]:
class BreakPredictor(nn.Module):
    def __init__(self, n_mels=128):
        super().__init__()

        self.cnn = nn.Sequential(
            # Layer 1
            nn.Conv2d(1, 16, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1)),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2), padding=(0, 0)),
        )

        # Dynamic feature size computation
        self._compute_feature_size(n_mels)

        # LSTM part
        self.lstm = nn.LSTM(
            input_size=self.feature_size,
            hidden_size=128,
            num_layers=1,  # 2
            batch_first=True,
            dropout=0.0  # 0.3
        )

        # Output layers
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def _compute_feature_size(self, n_mels):
        # Create a dummy input to compute the flattened size
        dummy_input = torch.zeros(1, 1, 1, n_mels)
        with torch.no_grad():
            try:
                cnn_out = self.cnn(dummy_input)
                self.feature_size = cnn_out.numel() // cnn_out.size(0)
                print(f"Feature size: {self.feature_size}")
                print(f"CNN output shape: {cnn_out.shape}")
            except Exception as e:
                print(f"Feature size computation failed: {e}")
                # Fallback computation
                self.feature_size = n_mels * 16  # Adjust based on your input

    def forward(self, x, lengths):
        batch_size, channels, time_steps, mels = x.size()

        # Ensure single channel input
        if channels > 1:
            x = x[:, 0:1, :, :]

        # Process each time step through CNN
        cnn_out = []
        for t in range(time_steps):
            step_input = x[:, :, t:t+1, :]

            # Add a check to handle very small inputs
            if step_input.size(-1) < 3:
                print(f"Warning: Input too small at time step {t}. Skipping.")
                continue

            out = self.cnn(step_input)
            cnn_out.append(out.flatten(1))

        # Check if we have any valid time steps
        if not cnn_out:
            raise ValueError("No valid time steps could be processed")

        # Stack time steps
        cnn_out = torch.stack(cnn_out, dim=1)

        # Adjust lengths based on processed time steps
        lengths = torch.tensor([min(len(cnn_out[0]), l) for l in lengths])

        # Pack padded sequence
        packed_input = nn.utils.rnn.pack_padded_sequence(
            cnn_out, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        # LSTM processing
        packed_output, _ = self.lstm(packed_input)

        # Unpack sequence
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)

        # Final classification
        output = self.fc(lstm_out)
        return output.squeeze(-1)

Feature size: 256
CNN output shape: torch.Size([1, 32, 1, 8])


In [ ]:
dataset = BreakDataset()
dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

model = BreakPredictor(n_mels=32)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters())

# Training loop
def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    for spectrograms, labels, lengths in dataloader:
        optimizer.zero_grad()
        outputs = model(spectrograms, lengths)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

In [26]:
train_epoch(model, dataloader, criterion, optimizer)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (256x40000 and 256x512)